# Mean-Variance Optimization (MVO): Textbook & Reference

**Instructions:** Read through this notebook and take notes. It contains the complete theory and fully functional code for standard Mean-Variance Optimization (Markowitz Portfolio Theory).

## Section 1: The Mathematics of MPT

Modern Portfolio Theory (MPT) assumes that investors are risk-averse, meaning that given two portfolios that offer the same expected return, investors will prefer the less risky one. Thus, an investor will take on increased risk only if compensated by higher expected returns.

### 1.1 Expected Portfolio Return
The expected return of a portfolio is calculated as a weighted sum of the individual assets' returns:
$$ E(R_p) = \sum_{i=1}^{n} w_i E(R_i) = \mathbf{w}^T \boldsymbol{\mu} $$
Where:
*   $E(R_p)$ is the expected return on the portfolio.
*   $w_i$ is the weight of asset $i$ in the portfolio.
*   $E(R_i)$ is the expected return of asset $i$.
*   $\mathbf{w}$ is the vector of portfolio weights.
*   $\boldsymbol{\mu}$ is the vector of expected asset returns.

### 1.2 Portfolio Variance (Risk)
The portfolio variance is a function of the variances of each asset and the covariances between them. This is the mathematical foundation of diversification.
$$ \sigma_p^2 = \sum_{i=1}^{n} \sum_{j=1}^{n} w_i w_j Cov(R_i, R_j) = \mathbf{w}^T \mathbf{\Sigma} \mathbf{w} $$
Where:
*   $\sigma_p^2$ is the portfolio variance.
*   $Cov(R_i, R_j)$ is the covariance between asset $i$ and asset $j$.
*   $\mathbf{\Sigma}$ is the covariance matrix of asset returns.

### 1.3 The Sharpe Ratio
The Sharpe Ratio measures the performance of an investment compared to a risk-free asset, after adjusting for its risk.
$$ SR = \frac{E(R_p) - R_f}{\sigma_p} $$
Where:
*   $SR$ is the Sharpe Ratio.
*   $R_f$ is the risk-free rate of return.
*   $\sigma_p$ is the standard deviation (volatility) of the portfolio (i.e., $\sqrt{\sigma_p^2}$).

### 1.4 Analytical Solution using Lagrange Multipliers (System of Equations)

If we want to find the portfolio with the absolute lowest risk (the **Global Minimum Variance** portfolio) and our *only* constraint is that the weights sum to 1, we can solve for the weights analytically. We first set up our objective, our constraint, and finally combine them into the Lagrangian equation.

$$ \text{Objective (Minimize Variance):} \quad \min_{\mathbf{w}} \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} w_i w_j \sigma_{ij} $$
$$ \text{Constraint (Weights sum to 1):} \quad \sum_{i=1}^{n} w_i = 1 $$
$$ \text{Lagrangian Equation:} \quad \mathcal{L} = \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} w_i w_j \sigma_{ij} - \lambda \left(\sum_{i=1}^{n} w_i - 1\right) $$

Instead of using matrix calculus right away, we take the partial derivative with respect to each individual weight $w_k$ and set it to zero. This gives us a system of $n$ equations:
$$ \frac{\partial \mathcal{L}}{\partial w_1} = \sum_{j=1}^{n} w_j \sigma_{1j} - \lambda = 0 $$
$$ \frac{\partial \mathcal{L}}{\partial w_2} = \sum_{j=1}^{n} w_j \sigma_{2j} - \lambda = 0 $$
$$ \vdots $$
$$ \frac{\partial \mathcal{L}}{\partial w_n} = \sum_{j=1}^{n} w_j \sigma_{nj} - \lambda = 0 $$

We can rewrite this system of linear equations cleanly in matrix form, where $\mathbf{\Sigma}$ is the covariance matrix and $\mathbf{1}$ is a vector of ones:
$$ \mathbf{\Sigma} \mathbf{w} - \lambda \mathbf{1} = \mathbf{0} $$
$$ \mathbf{\Sigma} \mathbf{w} = \lambda \mathbf{1} $$

Now, isolating for the weights vector $\mathbf{w}$ by multiplying both sides by the inverse covariance matrix ($\mathbf{\Sigma}^{-1}$):
$$ \mathbf{w} = \lambda \mathbf{\Sigma}^{-1} \mathbf{1} $$

We have an expression for $\mathbf{w}$, but we still need to find $\lambda$. We substitute our expression for $\mathbf{w}$ into our original constraint equation ($\mathbf{1}^T \mathbf{w} = 1$):
$$ \mathbf{1}^T (\lambda \mathbf{\Sigma}^{-1} \mathbf{1}) = 1 $$
$$ \lambda (\mathbf{1}^T \mathbf{\Sigma}^{-1} \mathbf{1}) = 1 \implies \lambda = \frac{1}{\mathbf{1}^T \mathbf{\Sigma}^{-1} \mathbf{1}} $$

Finally, substituting $\lambda$ back into our equation for $\mathbf{w}$ gives us the closed-form optimal weights vector $\mathbf{w}^*$:
$$ \mathbf{w}^* = \frac{\mathbf{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^T \mathbf{\Sigma}^{-1} \mathbf{1}} $$

## Section 2: Data Download\nWe will download historical price data for a sample universe of stocks.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Sample Universe
tickers = ['AAPL', 'MSFT', 'JPM', 'XOM', 'PG']

print("Downloading data...")
data = yf.download(tickers, start="2020-01-01", end="2024-01-01")['Close']
data = data.dropna()

# Calculate daily returns
returns = data.pct_change().dropna()
print("Data downloaded and returns calculated.")
returns.head()

## Section 3: Estimating Parameters\nWe need to estimate the expected returns ($\boldsymbol{\mu}$) and the covariance matrix ($\mathbf{\Sigma}$). We typically annualize these metrics.

In [ ]:
# Annualized Expected Returns (assuming 252 trading days)
mean_returns = returns.mean() * 252

# Annualized Covariance Matrix
cov_matrix = returns.cov() * 252

print("Expected Annual Returns:")
print(mean_returns)
print("\nCovariance Matrix:")
print(cov_matrix)

## Section 4: Optimization Setup\nWe want to find the weights $\mathbf{w}$ that maximize the Sharpe Ratio. Since standard optimizers *minimize* a function, we will minimize the *negative* Sharpe Ratio.

In [ ]:
risk_free_rate = 0.02
num_assets = len(tickers)

# Objective Function: Negative Sharpe Ratio
def negative_sharpe(weights, mean_returns, cov_matrix, risk_free_rate):
    # E(Rp) = w^T * mu
    portfolio_return = np.sum(mean_returns * weights)
    
    # sigma_p = sqrt(w^T * Sigma * w)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    # SR = (E(Rp) - Rf) / sigma_p
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return -sharpe_ratio

# Constraints: 
# 1. Weights must sum to 1
# 2. Weights must be between 0 and 1 (long-only)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0.0, 1.0) for _ in range(num_assets))

# Initial guess: Equal weighting
init_guess = num_assets * [1. / num_assets,]

## Section 5: Running the Optimizer\nWe use Sequential Least Squares Programming (SLSQP) to find the optimal weights.

In [ ]:
# Optimize
optimal_result = minimize(negative_sharpe, init_guess, args=(mean_returns, cov_matrix, risk_free_rate),
                          method='SLSQP', bounds=bounds, constraints=constraints)

opt_weights = optimal_result.x

print("Optimal Weights (Max Sharpe Portfolio):")
for i, ticker in enumerate(tickers):
    print(f"{ticker}: {opt_weights[i]:.2%}")
    
opt_return = np.sum(mean_returns * opt_weights)
opt_vol = np.sqrt(np.dot(opt_weights.T, np.dot(cov_matrix, opt_weights)))
print(f"\nExpected Return: {opt_return:.2%}")
print(f"Expected Volatility: {opt_vol:.2%}")
print(f"Sharpe Ratio: {(opt_return - risk_free_rate) / opt_vol:.2f}")

## Section 6: The Efficient Frontier\nWe can simulate thousands of random portfolios to visualize the Efficient Frontier and see where our optimal portfolio lies.

In [ ]:
# Simulate Random Portfolios
num_portfolios = 5000
results = np.zeros((3, num_portfolios))

for i in range(num_portfolios):
    # Random weights that sum to 1
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    
    # Portfolio Return & Volatility
    pret = np.sum(mean_returns * weights)
    pvol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    # Store Return, Volatility, and Sharpe Ratio
    results[0,i] = pvol
    results[1,i] = pret
    results[2,i] = (pret - risk_free_rate) / pvol

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(results[0,:], results[1,:], c=results[2,:], cmap='viridis', marker='o', s=10, alpha=0.5)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(opt_vol, opt_return, color='red', marker='*', s=300, label='Max Sharpe Portfolio')
plt.title('Efficient Frontier')
plt.xlabel('Expected Volatility (Risk)')
plt.ylabel('Expected Return')
plt.legend()
plt.show()